# Second Brain — Google Drive + Claude Sonnet 5

This notebook turns a Google Drive folder into a living "second brain": a personal knowledge vault that doesn't just store notes, but actively grows itself. The underlying idea comes from the CODE method (Capture, Organize, Distill, Express), popularized by productivity writers like Tiago Forte, which describes how a good external memory system should work. Traditionally this is a manual discipline — you capture ideas, tag and file them, periodically summarize clusters of notes, and turn distilled insight into new output. Here, Claude Sonnet 5 automates the Distill and Express stages, and does so with full awareness of what already exists in your vault, rather than working from a blank context window each time.

The notebook is organized as a progression. The first cells set up authentication to Google Drive and Claude's API, and establish basic read/write primitives: searching notes by keyword, reading their content regardless of file type, and writing new notes back into the folder. From there, two increasingly capable patterns are built: a simple Distill/Express pair that summarizes matching notes into a new note, and then a full self-reinforcing loop (`grow_node`) that takes a raw idea, searches the vault for related concepts, asks Claude to explicitly connect the idea to existing notes using Obsidian-style `[[wikilinks]]`, and writes the result back — including real backlinks inserted into the notes it connects to, and a plain-language explanation of how Claude reasoned through the vault to build those connections.

Because every new node is written back into the same folder that gets searched next time, the vault compounds: each idea you run through the loop makes the next one smarter, since it has more context to draw on. This mirrors, in miniature, the role MCP (Model Context Protocol) plays in giving AI models structured, standardized access to external tools and data — except here the "tool" is your own thinking, growing over time inside your Drive folder.

## Cell 1 — Install + Auth

This cell prepares the two connections everything else depends on: Google Drive and the Claude API. It first installs the `anthropic` Python package, since Colab environments don't include it by default. Next, it runs Colab's built-in `auth.authenticate_user()`, which opens a Google sign-in flow tied to your own account — this is simpler than a standard OAuth setup because Colab handles the consent screen and token management for you, with no `credentials.json` file to manage. Those credentials are then used to build a Drive API client (`drive`), which every later cell relies on to search, read, and write notes.

The second half of the cell sets up the Claude connection. It reads your Anthropic API key from Colab's Secrets manager via `userdata.get("ANTHROPIC_API_KEY")`, rather than hardcoding it in the notebook — this keeps the key out of the notebook file itself, so it's safe to share or version-control the notebook without leaking credentials. It also defines `CLAUDE_MODEL = "claude-sonnet-5"` as a single constant used by every function that calls Claude later, so the model choice only needs to be changed in one place. If this cell fails, nearly everything downstream will fail too, so it's worth confirming the printed "Auth OK" message before continuing.

In [ ]:
!pip install -q anthropic

from google.colab import auth, userdata
auth.authenticate_user()

from googleapiclient.discovery import build
import google.auth

creds, _ = google.auth.default()
drive = build("drive", "v3", credentials=creds)

import anthropic
client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))

CLAUDE_MODEL = "claude-sonnet-5"

print("Auth OK. Drive and Claude client ready.")

## Cell 2 — Vault folder

This cell tells the rest of the notebook which Google Drive folder is your actual vault, and immediately verifies that the connection works. `VAULT_FOLDER_ID` is extracted from the shareable Drive link you provided — it's the string of characters after `/folders/` in the URL, which uniquely identifies that folder regardless of its name or location. Every search, read, and write function later in the notebook is scoped to this one folder ID, so nothing outside your vault is ever touched.

The second part of the cell performs a live sanity check: it queries Drive for every non-trashed file whose parent is that folder, and prints each one's name and MIME type. This matters for two reasons. First, it confirms the folder ID is correct and that your Colab session actually has permission to see its contents — if the list comes back empty when you expect notes, that's an early signal something is misconfigured (wrong ID, wrong account, or a sharing permission issue), rather than a confusing failure several cells later. Second, seeing the MIME types up front is useful context, since your vault may contain a mix of native Google Docs and plain markdown/text files, which later cells (particularly `read_note` and the backlink-writing logic) need to treat differently.

In [ ]:
# From: https://drive.google.com/drive/folders/1_G0hxnITjO1lkCVq-trfQJdzi004BQhO
VAULT_FOLDER_ID = "1_G0hxnITjO1lkCVq-trfQJdzi004BQhO"

# Sanity check: list what's in the vault right now
check = drive.files().list(
    q=f"'{VAULT_FOLDER_ID}' in parents and trashed = false",
    fields="files(id, name, mimeType)"
).execute().get("files", [])

print(f"Found {len(check)} item(s) in vault:")
for f in check:
    print(f" - {f['name']}  ({f['mimeType']})")

## Cell 3 — Search + Read notes

This cell defines the two most fundamental operations the whole system depends on: finding notes, and reading them. `search_notes(query)` builds a Drive API query using `fullText contains`, which performs a keyword search across file contents (not just filenames) restricted to your vault folder. It's a literal, exact-text match rather than a semantic or fuzzy search — good enough for a personal vault of moderate size, but worth knowing as a limitation if the vault grows very large or if you search using different wording than the notes themselves use.

`read_note(file_id, mime_type)` handles a subtlety of Google Drive: notes can exist either as native Google Docs (which have no raw downloadable text and must be *exported* to plain text) or as ordinary files like `.md` or `.txt` (which can be downloaded directly as bytes and decoded). This function checks the MIME type and routes to the correct Drive API call accordingly, so every other function in the notebook can call `read_note` without needing to know or care which type of file it's dealing with. Getting this abstraction right early is what lets `distill`, `express`, and `grow_node` later treat the vault as one uniform collection of readable text, regardless of how each note happens to be stored.

In [ ]:
def search_notes(query: str):
    """Search note titles/content in the vault folder."""
    q = f"'{VAULT_FOLDER_ID}' in parents and fullText contains '{query}' and trashed = false"
    results = drive.files().list(q=q, fields="files(id, name, mimeType)").execute()
    return results.get("files", [])


def read_note(file_id: str, mime_type: str = None) -> str:
    """Read a note's content. Handles both plain files and native Google Docs."""
    if mime_type == "application/vnd.google-apps.document":
        data = drive.files().export(fileId=file_id, mimeType="text/plain").execute()
        return data.decode("utf-8") if isinstance(data, bytes) else data
    else:
        data = drive.files().get_media(fileId=file_id).execute()
        return data.decode("utf-8") if isinstance(data, bytes) else data


# Example:
# search_notes("project ideas")

## Cell 4 — Capture a new note

This cell implements the "Capture" step of the CODE method: writing a brand-new note directly into your Drive vault. `capture_note(title, content)` builds a Drive file metadata object (name and parent folder), wraps the note's text in a `MediaInMemoryUpload` object, and calls the Drive API's `create` method to upload it. Using `MediaInMemoryUpload` rather than writing content to a temporary file on disk first is a small but deliberate efficiency choice — it keeps the whole operation in memory and avoids managing throwaway files in the Colab filesystem, which matters once this function starts being called repeatedly and automatically by later cells rather than by hand.

Every note this function creates is saved with a `.md` extension, so it opens correctly as markdown in Obsidian or any other markdown-aware editor if you point one at this same Drive folder later. This function is intentionally simple and dumb on its own — it doesn't check for duplicate titles, doesn't validate content, and doesn't know anything about tags or links. That's by design: it's a low-level primitive that later, smarter functions (`express`, and especially `grow_node`) build on top of, adding structure, frontmatter, and relationships before ultimately calling this same function to do the actual writing.

In [ ]:
from googleapiclient.http import MediaInMemoryUpload

def capture_note(title: str, content: str) -> str:
    """Create a new markdown note in the vault (the 'Capture' step)."""
    file_metadata = {"name": f"{title}.md", "parents": [VAULT_FOLDER_ID]}
    media = MediaInMemoryUpload(content.encode("utf-8"), mimetype="text/markdown")
    f = drive.files().create(body=file_metadata, media_body=media, fields="id, name").execute()
    return f"Saved: {f['name']} (id: {f['id']})"


# Example:
# capture_note("New idea", "Some captured thought here.")

## Cell 5 — Distill

This cell implements the "Distill" step: taking a scattered set of matching notes and compressing them into a single, coherent summary using Claude. `distill(query)` first calls `search_notes` to find every note relevant to the query, then reads each one's full content with `read_note` and concatenates them into one large context block, separated by `---` and labeled with each note's filename as a heading. This combined text becomes the body of a single message sent to Claude via `client.messages.create`, using the `CLAUDE_MODEL` constant defined in Cell 1.

The prompt itself is deliberately simple: ask Claude to summarize the key ideas and connections across the provided notes, organized by theme. This is where the LLM genuinely earns its place in the loop — rather than just concatenating or keyword-matching your notes, it identifies overlapping ideas, contradictions, and threads that span multiple notes, something a plain search index can't do. If no notes match the query at all, the function short-circuits and returns a plain "No matching notes found" message rather than sending an empty or near-empty prompt to Claude, which would waste an API call and likely produce an unhelpful, ungrounded response. This function returns plain text — it doesn't write anything back to the vault itself; that responsibility belongs to `express`, defined next.

In [ ]:
def distill(query: str) -> str:
    """Search the vault, then have Claude summarize/connect the matching notes."""
    matches = search_notes(query)
    if not matches:
        return "No matching notes found."

    combined = "\n\n---\n\n".join(
        f"# {m['name']}\n{read_note(m['id'], m.get('mimeType'))}"
        for m in matches
    )

    response = client.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=1000,
        messages=[{
            "role": "user",
            "content": (
                "Summarize the key ideas and connections across these notes. "
                "Be concise and organize by theme:\n\n" + combined
            )
        }]
    )
    return response.content[0].text


# Example:
# print(distill("second brain"))

## Cell 6 — Express

This cell implements the "Express" step, and in doing so closes the first, simplest version of the loop. `express(query, note_title)` is a thin wrapper: it calls `distill(query)` to get a synthesized summary from Claude, then immediately hands that summary to `capture_note` to save it back into the vault as a brand-new note. If no `note_title` is supplied, it auto-generates one from the query text, so this function can be called with a single argument in the common case.

The significance of this cell is structural rather than technical: it's the first point in the notebook where the vault writes to itself based on its own content, rather than only being written to by you directly. Running `express("some topic")` reads existing notes, has Claude synthesize them, and adds the synthesis back as a new note — which itself becomes searchable and readable by future calls to `distill` or `express`. This is a genuine feedback loop, just a shallow one: it summarizes what's already there without deliberately searching out new connections or explaining its reasoning. The next section (`grow_node`) builds directly on this same distill-then-capture pattern, but adds explicit relationship-building via wikilinks, bidirectional backlinks, and a transparent reasoning trace — turning a one-shot summarizer into a genuine, explorable knowledge graph.

In [ ]:
def express(query: str, note_title: str = None) -> str:
    """Distill matching notes, then save the synthesis as a new note (the 'Express' step)."""
    summary = distill(query)
    if summary == "No matching notes found.":
        return summary

    title = note_title or f"Synthesis - {query}"
    return capture_note(title, summary)


# Example:
# express("second brain", "Synthesis - Second Brain Concepts")

## Cell 7 — The Loop: idea → vault-aware analysis → new linked node → back into vault

This cell defines `grow_node`, the core of the self-reinforcing loop, along with three supporting helpers. `_list_all_notes` lists every note title currently in the vault so Claude knows exactly which notes it's allowed to link to — this constraint is important, since without it a language model will sometimes invent plausible-sounding links to notes that don't actually exist. `_update_note_content` and `get_backlinks` support real, verifiable bidirectional linking, covered in more detail below.

`grow_node(idea)` itself works in stages. First, it searches the vault for notes related to the idea and gathers the full vault's list of titles. Second, it sends Claude a structured prompt containing the idea, the related notes' full content, and the complete list of valid link targets, asking for a JSON response containing a title, tags, outgoing links, the note's markdown content with inline `[[wikilinks]]`, and — critically — a `reasoning` field explaining which notes it considered, which it judged relevant, and why. Third, it assembles this into a properly formatted markdown file with Obsidian-style YAML frontmatter and saves it via `capture_note`. Fourth, and this is what makes the graph genuinely bidirectional rather than one-directional, it goes back into each note the new node links to and physically inserts a backlink pointing to the new node, then re-verifies all backlinks by scanning the whole vault for actual `[[title]]` occurrences — ground truth, not just what the LLM claims it did.

In [ ]:
import json
import re
from datetime import datetime

def _list_all_notes():
    """List every note currently in the vault (title only), so Claude knows what it can link to."""
    results = drive.files().list(
        q=f"'{VAULT_FOLDER_ID}' in parents and trashed = false",
        fields="files(id, name, mimeType)"
    ).execute().get("files", [])
    return results


def _slugify(title: str) -> str:
    slug = re.sub(r"[^\w\s-]", "", title).strip().replace(" ", "-")
    return slug[:80] if slug else "untitled"


def _update_note_content(file_id: str, new_content: str, mime_type: str = None):
    """Overwrite an existing plain-text/markdown file's content in place."""
    if mime_type == "application/vnd.google-apps.document":
        raise ValueError("Editing native Google Docs in place isn't supported here; only .md/text files.")
    media = MediaInMemoryUpload(new_content.encode("utf-8"), mimetype="text/markdown")
    drive.files().update(fileId=file_id, media_body=media).execute()


def get_backlinks(note_title: str) -> list:
    """
    Scan the WHOLE vault and return every note that contains a [[note_title]] link.
    This is the real, ground-truth backlink list (like Obsidian's backlink panel),
    computed fresh each time rather than trusted from what the LLM claims.
    """
    backlinks = []
    for n in _list_all_notes():
        try:
            content = read_note(n["id"], n.get("mimeType"))
        except Exception:
            continue
        if f"[[{note_title}]]" in content:
            backlinks.append(n["name"])
    return backlinks


def grow_node(idea: str, related_query: str = None, write_backlinks: bool = True) -> dict:
    """
    One full turn of the loop:
      1. List the whole vault + pull full content of keyword-related notes.
      2. Ask Claude to analyze the idea against them, and to EXPLAIN its reasoning.
      3. Get back {title, tags, links, content, reasoning}.
      4. Write the new node as .md with Obsidian frontmatter + [[wikilinks]].
      5. (optional) Go BACK into each linked note and insert a real backlink to the new node,
         so the connection is visible from both sides on disk, not just in one direction.
      6. Verify backlinks by re-scanning the vault (ground truth, not just what Claude said).
    """
    all_notes = _list_all_notes()
    note_titles = [n["name"].rsplit(".", 1)[0] for n in all_notes]

    q = related_query or idea
    related = search_notes(q)
    related_context = "\n\n---\n\n".join(
        f"# {n['name']}\n{read_note(n['id'], n.get('mimeType'))}"
        for n in related
    ) or "(no closely related notes found by keyword search)"

    prompt = f"""You are helping grow a personal knowledge vault (Obsidian-style, markdown with [[wikilinks]]).

NEW IDEA TO INTEGRATE:
{idea}

EXISTING NOTE TITLES IN THE VAULT (only link to these, using exact names):
{json.dumps(note_titles)}

RELATED NOTES FOUND BY KEYWORD SEARCH (full content, for context):
{related_context}

Analyze the idea in light of the existing vault. Produce a NEW note that:
- Distills the idea clearly
- Explicitly connects it to relevant existing notes using [[Exact Note Title]] wikilinks inline in the prose, wherever a genuine conceptual link exists
- Only links to titles that appear in the list above (never invent a link to a note that doesn't exist)
- Adds a short "Relationships" section at the end listing each link and WHY it connects

Respond with ONLY valid JSON, no markdown fences, no preamble, in this exact shape:
{{
  "title": "short descriptive title",
  "tags": ["tag1", "tag2"],
  "links": ["Exact Note Title 1", "Exact Note Title 2"],
  "content": "full markdown body, including inline [[wikilinks]] and the Relationships section",
  "reasoning": "step-by-step explanation: which search terms/notes you looked at, which ones you judged relevant and why, which ones you rejected and why, and how you decided on each link"
}}"""

    response = client.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=2000,
        messages=[{"role": "user", "content": prompt}]
    )

    raw = response.content[0].text.strip()
    raw = re.sub(r"^```json\s*|\s*```$", "", raw.strip())

    try:
        node = json.loads(raw)
    except json.JSONDecodeError:
        raise ValueError(f"Claude did not return valid JSON:\n{raw}")

    frontmatter = (
        "---\n"
        f"title: {node['title']}\n"
        f"created: {datetime.utcnow().isoformat()}Z\n"
        f"tags: [{', '.join(node.get('tags', []))}]\n"
        f"links: [{', '.join('[[' + l + ']]' for l in node.get('links', []))}]\n"
        "---\n\n"
    )
    full_md = frontmatter + node["content"]

    save_result = capture_note(node["title"], full_md)

    # --- Write real backlinks into the notes this new node points to ---
    backlink_writes = []
    if write_backlinks:
        by_title = {n["name"].rsplit(".", 1)[0]: n for n in all_notes}
        for link_title in node.get("links", []):
            target = by_title.get(link_title)
            if not target:
                continue
            try:
                existing_content = read_note(target["id"], target.get("mimeType"))
                marker = f"[[{node['title']}]]"
                if marker in existing_content:
                    continue  # already linked
                if "## Backlinks" in existing_content:
                    updated = existing_content.rstrip() + f"\n- {marker}\n"
                else:
                    updated = existing_content.rstrip() + f"\n\n## Backlinks\n- {marker}\n"
                _update_note_content(target["id"], updated, target.get("mimeType"))
                backlink_writes.append(link_title)
            except ValueError:
                # native Google Doc — can't edit in place with this simple approach
                pass

    # Ground-truth backlinks for the new node (will grow as future ideas link to it)
    verified_backlinks = get_backlinks(node["title"])

    return {
        "title": node["title"],
        "tags": node.get("tags", []),
        "links": node.get("links", []),
        "reasoning": node.get("reasoning", ""),
        "backlinks_written_into": backlink_writes,
        "backlinks_verified": verified_backlinks,
        "save_result": save_result,
        "content": full_md,
    }

## Cell 7b — Display helper: show markdown, backlinks, and the LLM's exploration reasoning

This cell defines a single function, `show_node`, whose only job is presentation: taking the dictionary returned by `grow_node` and printing it in a clearly organized, human-readable format, rather than leaving you to inspect a raw Python dict. It exists purely because `grow_node` returns a lot of information at once — the note's title and tags, the LLM's reasoning, outgoing links, backlinks written into other notes, verified backlinks from a fresh vault scan, and the full markdown body — and dumping all of that as a single unstructured print statement would be hard to actually read and evaluate.

The function prints each of these in its own clearly labeled section, in a deliberate order: identity (title/tags) first, then the LLM's reasoning (so you can audit *how* it explored the vault before seeing what it produced), then outgoing links, then the two kinds of backlink information side by side — what was actually written into other files versus what a fresh scan of the vault confirms exists — and finally the complete raw markdown of the new note itself, exactly as it was saved to Drive. Separating "backlinks written" from "backlinks verified" is intentional: it lets you catch cases where a link was claimed but the corresponding file couldn't be updated (for example, if it's a native Google Doc rather than a plain text file).

In [ ]:
def show_node(result: dict):
    """Pretty-print everything about a node created by grow_node()."""
    print("=" * 70)
    print(f"TITLE: {result['title']}")
    print(f"TAGS:  {result['tags']}")
    print("=" * 70)

    print("\n--- LLM'S EXPLORATION / REASONING ---\n")
    print(result["reasoning"] or "(no reasoning returned)")

    print("\n--- OUTGOING LINKS (this note -> existing notes) ---")
    for l in result["links"]:
        print(f"  -> [[{l}]]")

    print("\n--- BACKLINKS WRITTEN INTO EXISTING NOTES (existing note -> this new node) ---")
    if result["backlinks_written_into"]:
        for l in result["backlinks_written_into"]:
            print(f"  [[{l}]]  now contains a link back to  [[{result['title']}]]")
    else:
        print("  (none written — either no links resolved to files, or already present)")

    print("\n--- VERIFIED BACKLINKS (ground-truth scan of the whole vault right now) ---")
    if result["backlinks_verified"]:
        for b in result["backlinks_verified"]:
            print(f"  <- {b}")
    else:
        print("  (none yet — nothing in the vault currently links to this node)")

    print("\n--- FULL MARKDOWN OF THE NEW NOTE ---\n")
    print(result["content"])
    print("\n" + "=" * 70)

## Cell 8a — Run the loop on a single idea

This cell runs the entire loop end-to-end on one concrete example idea, so you can see the whole system work before adapting it to your own ideas. Calling `grow_node("Idea: habits are just cached decisions the brain stopped re-evaluating.")` triggers everything described in Cell 7: a vault search for related notes, a call to Claude with full vault context, parsing of the structured JSON response, writing the new note to Drive, inserting backlinks into whatever existing notes it connected to, and re-verifying those backlinks by scanning the vault. The result is a dictionary containing all of that information.

That dictionary is then passed straight into `show_node`, defined in the previous cell, which prints it out in full: the LLM's reasoning about how it explored the vault, the links it created, the backlinks it wrote and verified, and the complete markdown of the new note. This is the cell to run first when testing changes to the prompt or logic inside `grow_node` — it's cheap to re-run, produces a single new note each time, and gives you immediate, complete visibility into what the model actually did, rather than requiring you to separately go check the Drive folder by hand.

In [ ]:
# Single idea through the loop:
result = grow_node("Idea: habits are just cached decisions the brain stopped re-evaluating.")
show_node(result)

## Cell 8b — Run the loop on a chain of ideas

This cell demonstrates the compounding property of the loop by running several ideas through `grow_node` in sequence, one after another. The key thing to notice is that nothing special is done to connect these ideas manually — `grow_node` re-lists the entire vault and re-runs its search at the start of every single call, so the second idea in the list is analyzed with full awareness of the node the first idea just created, and would be able to link to it if genuinely relevant. This is what makes the system a real loop rather than a batch of independent, parallel operations: each iteration's output becomes part of the next iteration's input, entirely through the shared state of the Drive folder itself.

For each idea, `show_node` is called immediately after `grow_node`, so you see the full breakdown — reasoning, links, backlinks, markdown — printed for every node in the chain as it's created, rather than only at the end. This is useful for watching the graph grow in real time and catching any point where the model's linking or reasoning starts to drift, before running a longer chain of ideas unattended.

In [ ]:
# Chain of ideas: each one sees the nodes created by the ones before it,
# because grow_node() re-lists the vault every call.

ideas = [
    "Idea: attention is the real currency of a second brain, not storage.",
    "Idea: forgetting on purpose (pruning notes) might be as important as capturing.",
]

for idea in ideas:
    r = grow_node(idea)
    show_node(r)

## Quick start

Run cells 1-8 in order once to define every function. After that, you can call any of these directly in the playground cell below, or in new cells you add:

```python
search_notes("your query")
capture_note("Title", "Content...")
distill("your query")
express("your query")
grow_node("your idea")   # the full loop: idea -> linked node -> backlinks -> back into vault
show_node(result)        # pretty-print a grow_node() result
get_backlinks("Some Note Title")  # ground-truth backlink scan for any note
```

## Playground

This final cell is intentionally left empty as a scratch space for your own experimentation, once every function above has been defined by running the earlier cells. Rather than editing the core functions directly to try things out, use this cell to call `grow_node` with your own ideas, inspect intermediate results with `show_node`, or call individual pieces like `search_notes` or `get_backlinks` on their own to explore the current state of your vault.

Because every function defined earlier in the notebook is now available in this same Python session, you can freely mix and match them here — for example, running `get_backlinks("Some Note Title")` on a note you created days ago to see what has since linked to it, or calling `distill` on a broad topic before deciding whether to commit a full `grow_node` call. Treat this cell (and any you add after it) as the actual working surface of the notebook; the cells above it are the engine, and this is where you drive it. If you find yourself repeating the same multi-line pattern here often, that's usually a sign it's worth turning into its own named function further up, the same way `express` and `grow_node` were built from simpler pieces.

In [ ]:
# Your playground cell
